# Cribador ADK — fidelidad vs CX (en GPU Kaggle, $0)

Corre el MISMO modelo que en local (`qwen2.5:14b` q4 vía Ollama) sobre GPU T4 → **fidelidad idéntica**, hardware más rápido (sobre todo el *prefill* del prompt de ~32k).

**Antes de ejecutar:**
1. `Settings → Accelerator → GPU T4 x2`
2. `Settings → Internet → ON` (para `pip`, `ollama pull` y el webhook público)
3. Sube el bundle (`package_for_kaggle.sh`) como **Dataset** y adjúntalo (`Add Input`).


## 1 · Ollama + GPU

In [ ]:
import os, subprocess, time, shutil
# El instalador de Ollama necesita zstd para descomprimir; la imagen Kaggle no lo trae.
subprocess.run('apt-get update -qq && apt-get install -y -qq zstd', shell=True)
r = subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, capture_output=True, text=True)
print('install rc:', r.returncode); print(r.stdout[-600:]); print(r.stderr[-600:])
# El binario suele ir a /usr/local/bin, pero ese dir puede NO estar en el PATH del kernel Kaggle.
ollama = shutil.which('ollama') or '/usr/local/bin/ollama'
assert os.path.exists(ollama), f'ollama NO instalado en {ollama}'
os.environ['PATH'] = os.path.dirname(ollama) + ':' + os.environ.get('PATH', '')
os.environ['OLLAMA_FLASH_ATTENTION'] = '1'
os.environ['OLLAMA_CONTEXT_LENGTH'] = '32768'   # el prompt de Petal (~32k) cabe sin truncar
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
subprocess.Popen([ollama, 'serve'], env={**os.environ})
time.sleep(8)
print('ollama en', ollama, '· serve lanzado')

In [ ]:
# Descarga del modelo (~9 GB, requiere Internet ON). Tarda unos minutos la 1a vez.
# PATH + OLLAMA_HOST ya quedaron en os.environ (celda anterior) → los hereda el shell.
!ollama pull qwen2.5:14b
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
!ollama list

## 2 · Dependencias Python

In [ ]:
!pip install -q google-adk litellm google-genai pyyaml requests

## 3 · Reconstruir el repo desde el Dataset
Auto-detecta el bundle dentro de `/kaggle/input` (sin importar el slug del dataset).

In [ ]:
import glob, os, shutil, zipfile
DST = '/kaggle/working/repo'
if os.path.exists(DST): shutil.rmtree(DST)
# Kaggle puede dejar el dataset como ZIP o ya extraído → soportamos ambos.
zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
if zips:
    with zipfile.ZipFile(zips[0]) as z: z.extractall('/kaggle/working/_extract')
    base = '/kaggle/working/_extract'
else:
    base = '/kaggle/input'
cands = glob.glob(base + '/**/definitions', recursive=True)
assert cands, f'No encuentro definitions/ en {base} — ¿adjuntaste el Dataset?'
SRC = os.path.dirname(cands[0])
shutil.copytree(SRC, DST)
if not os.path.exists(DST + '/.env'):
    open(DST + '/.env', 'w').write('GEMINI_API_KEY=unused-on-kaggle\n')
print('repo →', DST, '·', sorted(os.listdir(DST)))

## 4 · Correr la fidelidad
`ADK_RECON=multi` → multi-agente. Quítalo para la reconstrucción plana.
`--limit N` para una prueba corta; sin flag = los 51 TCs.

In [ ]:
RECON = 'multi'          # 'multi' | 'flat'
LIMIT = ''               # ej. '--limit 8' para prueba corta; '' = los 51
env = ('ADK_RECON=multi ' if RECON == 'multi' else '')
cmd = (f'cd /kaggle/working/repo && OLLAMA_API_BASE=http://127.0.0.1:11434 '
       f'OLLAMA_FLASH_ATTENTION=1 OLLAMA_CONTEXT_LENGTH=32768 {env}'
       f'python -u qap/adk_fidelity/run_fidelity.py {LIMIT}')
print(cmd)
get_ipython().system(cmd)

## 5 · Descargar el resultado

In [ ]:
from IPython.display import FileLink
p = '/kaggle/working/repo/qap/adk_fidelity/fidelity_result.json'
import shutil; shutil.copy(p, '/kaggle/working/fidelity_result.json')
FileLink('/kaggle/working/fidelity_result.json')